# Population EDA — Points vs Minutes (60-minute Boundary)

Visualises `total_points` vs `minutes` for every player-gameweek in the mart,
colour-coded by population membership:

- **Blue** — minutes ≥ 60 (`filter_performance`): the performance population used in all signal characterisation.
- **Red** — 0 < minutes < 60: appearances below the threshold, excluded from the performance population.

The vertical dashed line marks the 60-minute boundary (`CLEAN_SHEET_MIN_MINUTES`). Dotted
horizontal lines show the mean `total_points` within each regime per position.

**Purpose:** Make the structural scoring-regime break at 60 minutes visually legible.
The distribution shift across the boundary is the empirical justification for `filter_performance`
as an analytical population — it is not an arbitrary filter.

This notebook is exploratory. It does not produce gate decisions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from dal.config import DB_PATH
from dal.exceptions import MartNotBuiltError, MartSchemaError
from dal.pipeline import load as load_mart, run as run_pipeline
from domain.fpl_scoring import CLEAN_SHEET_MIN_MINUTES

THRESHOLD = CLEAN_SHEET_MIN_MINUTES  # 60
POSITIONS = ["GK", "DEF", "MID", "FWD"]

COLOR_ABOVE = "#1f77b4"  # blue  — performance population (minutes >= 60)
COLOR_BELOW = "#d62728"  # red   — below threshold (0 < minutes < 60)

pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_columns", 20)

In [ ]:
try:
    result = load_mart(DB_PATH)
except (MartNotBuiltError, MartSchemaError) as e:
    print(f"Rebuilding mart ({type(e).__name__})...")
    run_pipeline(DB_PATH, force=True)
    result = load_mart(DB_PATH)

mart = result.mart.copy()

print(f"Mart: {len(mart):,} rows  |  GW {mart['gw'].min()}–{mart['gw'].max()}  |  cutoff GW {result.data_cutoff_gw}")
print(f"Positions: {sorted(mart['position'].unique())}")
print(f"Minutes range: {mart['minutes'].min()}–{mart['minutes'].max()}")

**FPL question:** Of all player-gameweeks, how many even matter for a starting decision?

**Why it matters:** Only ~24% of player-GWs are ≥60-minute "performance" rows; 56% are non-appearances and 11% are sub-60 cameos. This sizes the two populations the rest of the notebook compares — and flags that the `<60` group, while large overall, is position-dependent (almost no GKs).

In [ ]:
no_fixture = mart[mart["minutes"].isna()]   # BGW: club had no fixture this GW → minutes/points NULL
zero_min   = mart[mart["minutes"] == 0]      # club played, this player did not
below_60   = mart[(mart["minutes"] > 0) & (mart["minutes"] < THRESHOLD)]
above_60   = mart[mart["minutes"] >= THRESHOLD]

n = len(mart)
accounted = len(no_fixture) + len(zero_min) + len(below_60) + len(above_60)
print(f"Population breakdown (all {n:,} mart rows):")
print(f"  no fixture / BGW (minutes NULL)    : {len(no_fixture):>6,}  ({len(no_fixture)/n*100:5.1f}%)")
print(f"  minutes = 0  (no appearance)       : {len(zero_min):>6,}  ({len(zero_min)/n*100:5.1f}%)")
print(f"  0 < minutes < {THRESHOLD}  (below threshold) : {len(below_60):>6,}  ({len(below_60)/n*100:5.1f}%)")
print(f"  minutes >= {THRESHOLD}  (performance pop)    : {len(above_60):>6,}  ({len(above_60)/n*100:5.1f}%)")
print(f"  {'-' * 44}")
print(f"  total accounted                    : {accounted:>6,}  (should equal {n:,})")

**FPL question:** Is there a real scoring-regime break at 60 minutes, or is points just a smooth function of minutes?

**Why it matters:** The entire `filter_performance` population assumes 60 minutes is a *structural* threshold (2nd appearance point + clean-sheet eligibility), not an arbitrary cut. If the break is real, a sub-60 cameo can't be judged on the same yardstick as a full game. *(Cross-sectional and overplotted — the views further down add the temporal and distributional angles.)*

In [ ]:
# Restrict scatter to players who appeared (minutes > 0).
# Zero-minute rows have no scoring-regime membership — they are not part of either population.
played = mart[mart["minutes"] > 0].copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=False)
axes_flat = axes.flatten()

for i, pos in enumerate(POSITIONS):
    ax      = axes_flat[i]
    pos_df  = played[played["position"] == pos]
    below   = pos_df[pos_df["minutes"] < THRESHOLD]
    above   = pos_df[pos_df["minutes"] >= THRESHOLD]

    ax.scatter(
        below["minutes"], below["total_points"],
        color=COLOR_BELOW, alpha=0.40, s=12, zorder=2,
        label=f"< {THRESHOLD} min  (n={len(below):,})",
    )
    ax.scatter(
        above["minutes"], above["total_points"],
        color=COLOR_ABOVE, alpha=0.25, s=12, zorder=2,
        label=f"≥ {THRESHOLD} min  (n={len(above):,})",
    )

    # 60-minute boundary
    ax.axvline(THRESHOLD, color="black", linewidth=1.4, linestyle="--", zorder=3,
               label=f"{THRESHOLD}-min boundary")

    # Mean-points reference lines per regime
    if len(below):
        ax.axhline(below["total_points"].mean(), color=COLOR_BELOW,
                   linewidth=1.0, linestyle=":", alpha=0.9)
    if len(above):
        ax.axhline(above["total_points"].mean(), color=COLOR_ABOVE,
                   linewidth=1.0, linestyle=":", alpha=0.9)

    ax.set_title(pos, fontsize=13, fontweight="bold")
    ax.set_xlabel("minutes played")
    ax.set_ylabel("total_points")
    ax.xaxis.set_major_locator(mticker.MultipleLocator(30))
    ax.legend(fontsize=8, markerscale=1.8)
    ax.grid(alpha=0.2)

fig.suptitle(
    f"total_points vs minutes  —  60-minute population boundary  (minutes > 0)",
    fontsize=13, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.show()

**FPL question:** How many points does crossing 60 minutes add, per position?

**Why it matters:** Quantifies the regime gap (the "lift") as a single expected-value number — the headline case for treating ≥60 as its own population. It's a *mean* lift, though; the later views show what that average conceals.

In [ ]:
# Regime summary table: how much does crossing the 60-minute line shift mean points?
rows = []
for pos in POSITIONS:
    ab = above_60[above_60["position"] == pos]["total_points"]
    bl = below_60[below_60["position"] == pos]["total_points"]

    rows.append({
        "position"          : pos,
        f"n  (<{THRESHOLD}m)"   : len(bl),
        f"mean  (<{THRESHOLD}m)": round(bl.mean(), 2) if len(bl) else float("nan"),
        f"n  (≥{THRESHOLD}m)"   : len(ab),
        f"mean  (≥{THRESHOLD}m)": round(ab.mean(), 2) if len(ab) else float("nan"),
        "mean_lift"         : round(ab.mean() - bl.mean(), 2) if len(ab) and len(bl) else float("nan"),
        "zero_mass_above_%" : round((ab == 0).mean() * 100, 1) if len(ab) else float("nan"),
    })

summary = pd.DataFrame(rows)
print("Regime summary by position (mean_lift = mean pts above − below threshold):")
display(summary)

## Regime × points over gameweek blocks

The scatter above answers a *cross-sectional, structural* question (does crossing 60
minutes shift the points cloud?). It cannot show **how the two populations move relative
to `total_points` over the season**, and overplotting hides the distribution shape. The
views below answer that directly.

All views use the **played population** — `minutes > 0`, **single gameweeks only**
(double-gameweeks excluded: a DGW row bundles two matches, ~115 min / 5.5 pts, so it isn't
comparable to a single appearance) — split into two regimes:

- **`≥60` (blue)** — the performance population (`filter_performance`).
- **`<60` (red)** — sub-threshold appearances.

Gameweeks are grouped into **5-GW blocks** because the `<60` group is only ~24 rows per
gameweek per position — too thin for per-GW means. Blocks (~120–250 rows/cell) smooth that
noise while preserving the seasonal trajectory.

> **Caveat — GK:** the `<60` regime is essentially an *outfield* phenomenon. Goalkeepers
> play 90 or 0, so there are only ~10 sub-60 GK appearances all season (none in several
> blocks). Read the GK panel as noise; the signal is in DEF / MID / FWD.

> **Caveat — this is exposure, not a clean causal effect.** *Who* plays 60+ isn't random:
> it skews toward nailed, higher-quality players on stronger teams, while the `<60` group is
> a selected mix of late subs, early hooks (blowouts) and red-card / injury exits (which tank
> points for non-exposure reasons). So the ≥60 "lift" overstates the pure effect of extra
> minutes. Treat these views as **descriptive exposure**, not the causal value of 30 more
> minutes. Also: the *mean* counts occasional hauls (top 10% of appearances = 34% of all
> points), so compare it against the median/quantiles (View 5) and dominance (View 7) — e.g.
> the ≥60 forward edge is mostly hauls (mean +2.6 but typical-game +1), whereas DEF/MID is
> broad-based (mean ~2.8, typical-game +2).

In [ ]:
# Shared setup for the views below — the played population is:
#   minutes > 0   AND   single gameweeks only (is_dgw excluded).
# A double-GW row bundles two matches (~115 min, 5.5 pts) so it is not comparable to a
# single appearance and would inflate the ≥60 group. BGW rows (minutes NULL, no fixture)
# are dropped automatically by the minutes > 0 test.
_played_all = mart[mart["minutes"] > 0]
played = _played_all[~_played_all["is_dgw"].astype(bool)].copy()
n_dgw = len(_played_all) - len(played)
played["regime"] = np.where(played["minutes"] >= THRESHOLD, "ge60", "lt60")

BLOCK_SIZE = 5
played["block"] = ((played["gw"] - 1) // BLOCK_SIZE) + 1

# GW-range label per block, e.g. {1: "1–5", 2: "6–10", ...}
_br = played.groupby("block")["gw"].agg(gw_min="min", gw_max="max")
BLOCK_LABELS = {b: f"{r.gw_min}–{r.gw_max}" for b, r in _br.iterrows()}
BLOCKS = sorted(played["block"].unique())

REGIME_COLOR = {"ge60": COLOR_ABOVE, "lt60": COLOR_BELOW}
REGIME_LABEL = {"ge60": f"≥{THRESHOLD} min", "lt60": f"<{THRESHOLD} min"}

print(f"Played rows (minutes > 0, single GW): {len(played):,}  ({n_dgw} double-GW rows excluded)")
print(f"Blocks ({BLOCK_SIZE} GW each): " + ", ".join(f"B{b}={BLOCK_LABELS[b]}" for b in BLOCKS))

### View 1 — mean points over blocks (expected value)

**FPL question:** Does the ≥60-vs-`<60` *expected-points* gap hold constant across the season, or drift?

**Why it matters / when to use the mean:** This is the **expected-value** view — the only statistic that compounds into season totals, so it's the right lens for *squad accumulation* and "is this population structurally higher-scoring." A flat gap means the 60-min rule is stable all season, not an early-GW artefact. (Use the *other* views, not this one, for captaincy or consistency calls.)

In [ ]:
# View 1 — Mean total_points per regime over gameweek blocks (faceted by position).
# Lines = block means; shaded band = 95% CI of the mean. Missing blocks (e.g. GK <60)
# appear as gaps. Most direct read on "how the gap moves over the season".
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)

for i, pos in enumerate(POSITIONS):
    ax = axes.flatten()[i]
    pos_df = played[played["position"] == pos]
    for regime in ("ge60", "lt60"):
        g = (pos_df[pos_df["regime"] == regime].groupby("block")["total_points"]
             .agg(["mean", "std", "count"]).reindex(BLOCKS).astype(float))
        ci = 1.96 * g["std"] / np.sqrt(g["count"])
        ax.plot(g.index, g["mean"], marker="o", linewidth=1.8,
                color=REGIME_COLOR[regime], label=REGIME_LABEL[regime], zorder=3)
        ax.fill_between(g.index, g["mean"] - ci, g["mean"] + ci,
                        color=REGIME_COLOR[regime], alpha=0.15, zorder=1)
    ax.set_title(pos, fontsize=13, fontweight="bold")
    ax.set_ylabel("mean total_points")
    ax.set_ylim(bottom=0)
    ax.set_xticks(BLOCKS)
    ax.set_xticklabels([BLOCK_LABELS[b] for b in BLOCKS], rotation=45, fontsize=8)
    ax.set_xlabel("gameweek block")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=9)

fig.suptitle("View 1 — mean total_points per regime over GW blocks  (minutes > 0; band = 95% CI of mean)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### View 2 — distribution shape (what the mean hides)

**FPL question:** What does a *typical* return in each regime look like — and how often is it nothing?

**Why it matters / when relevant:** Shows the shape the mean erases: the zero/low-return mass. Relevant whenever you care about **what you'll usually get** rather than the average — e.g. a sub-60 cameo is a near-certain blank, not a "1.2-point" asset.

In [ ]:
# View 2 — Distribution shape of total_points by regime (faceted by position).
# Violin = density (note the fat base near 0 for <60: many sub-60 cameos return nothing);
# black line = mean. This is the scatter's job done right — the shift IS the distribution.
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for i, pos in enumerate(POSITIONS):
    ax = axes.flatten()[i]
    pos_df = played[played["position"] == pos]
    data = [pos_df[pos_df["regime"] == reg]["total_points"].astype(float).values
            for reg in ("lt60", "ge60")]
    parts = ax.violinplot(data, positions=[1, 2], showmeans=True, showextrema=False, widths=0.8)
    for body, c in zip(parts["bodies"], [COLOR_BELOW, COLOR_ABOVE]):
        body.set_facecolor(c)
        body.set_alpha(0.45)
    if "cmeans" in parts:
        parts["cmeans"].set_color("black")
    for x, reg in zip((1, 2), ("lt60", "ge60")):
        v = pos_df[pos_df["regime"] == reg]["total_points"]
        ax.annotate(f"mean {v.mean():.2f}\n{(v == 0).mean()*100:.0f}% zero  (n={len(v):,})",
                    xy=(x + 0.18, v.mean()), fontsize=8, va="center")
    ax.set_xticks([1, 2])
    ax.set_xticklabels([REGIME_LABEL["lt60"], REGIME_LABEL["ge60"]])
    ax.set_title(pos, fontsize=13, fontweight="bold")
    ax.set_ylabel("total_points")
    ax.grid(alpha=0.2, axis="y")

fig.suptitle("View 2 — total_points distribution by regime  (minutes > 0; violin = density, line = mean)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### View 3 — lift stability (season × position)

**FPL question:** Is the regime gap stable across *both* the season and every position at once?

**Why it matters / when relevant:** A compact stability check. Near-uniform colour along a row = the same 60-min logic applies to that position all season. Use it to trust the filter as a **global rule** instead of re-deriving it per position/block.

In [ ]:
# View 3 — Lift heatmap: mean(≥60) − mean(<60) total_points for each block × position.
# Near-uniform colour along a row = the regime gap is stable across the season.
# "—" cells (GK) = no sub-60 appearances in that block.
piv = played.groupby(["position", "block", "regime"])["total_points"].mean().unstack("regime")
piv["lift"] = piv["ge60"] - piv["lt60"]
lift_mat = piv["lift"].unstack("block").reindex(POSITIONS)
arr = lift_mat.to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(arr, aspect="auto", cmap="viridis")
midval = np.nanmean(arr)
for r in range(arr.shape[0]):
    for c in range(arr.shape[1]):
        v = arr[r, c]
        label = f"{v:.2f}" if not np.isnan(v) else "—"
        ax.text(c, r, label, ha="center", va="center", fontsize=9,
                color="white" if (np.isnan(v) or v < midval) else "black")
ax.set_xticks(range(len(BLOCKS)))
ax.set_xticklabels([BLOCK_LABELS[b] for b in BLOCKS], rotation=45, fontsize=9)
ax.set_yticks(range(len(POSITIONS)))
ax.set_yticklabels(POSITIONS)
ax.set_xlabel("gameweek block")
ax.set_title(f"View 3 — mean lift in total_points  (≥{THRESHOLD}  −  <{THRESHOLD} min)   by block × position   (minutes > 0)",
             fontsize=12, fontweight="bold")
fig.colorbar(im, ax=ax, label="mean lift (pts)")
plt.tight_layout()
plt.show()

### View 4 — distribution + movement + lift, together

**FPL question:** How do the full distributions *and* the gap move together over the season?

**Why it matters / when relevant:** Ties spread + trend + lift into one figure — shows whether the gap rides on the body of the distribution or the tails. Use when you want the whole story in a single panel.

In [ ]:
# View 4 — total_points box plots per block & regime, with the lift overlaid (dashed).
# Richest single view: distribution + movement + gap together. Outliers hidden for legibility.
# Empty groups (GK <60 in some blocks) are skipped rather than drawn.
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=True)
OFFSET = 0.18

for i, pos in enumerate(POSITIONS):
    ax = axes.flatten()[i]
    pos_df = played[played["position"] == pos]
    lt_data, ge_data, lift_vals = [], [], []
    for b in BLOCKS:
        lo = pos_df[(pos_df["block"] == b) & (pos_df["regime"] == "lt60")]["total_points"]
        hi = pos_df[(pos_df["block"] == b) & (pos_df["regime"] == "ge60")]["total_points"]
        lt_data.append(np.asarray(lo, dtype=float))
        ge_data.append(np.asarray(hi, dtype=float))
        lm, hm = lo.mean(), hi.mean()
        lift_vals.append(float(hm - lm) if (len(lo) and len(hi) and pd.notna(lm) and pd.notna(hm)) else np.nan)

    lt_pos = [b - OFFSET for b, d in zip(BLOCKS, lt_data) if len(d)]
    ge_pos = [b + OFFSET for b, d in zip(BLOCKS, ge_data) if len(d)]
    b_lt = ax.boxplot([d for d in lt_data if len(d)], positions=lt_pos, widths=0.3,
                      patch_artist=True, showfliers=False, medianprops=dict(color="black"))
    b_ge = ax.boxplot([d for d in ge_data if len(d)], positions=ge_pos, widths=0.3,
                      patch_artist=True, showfliers=False, medianprops=dict(color="black"))
    for box in b_lt["boxes"]:
        box.set_facecolor(COLOR_BELOW); box.set_alpha(0.5)
    for box in b_ge["boxes"]:
        box.set_facecolor(COLOR_ABOVE); box.set_alpha(0.5)

    ax.plot(BLOCKS, lift_vals, "k--o", linewidth=1.4, markersize=4, zorder=5)
    ax.set_title(pos, fontsize=13, fontweight="bold")
    ax.set_ylabel("total_points")
    ax.set_xticks(BLOCKS)
    ax.set_xticklabels([BLOCK_LABELS[b] for b in BLOCKS], rotation=45, fontsize=8)
    ax.set_xlabel("gameweek block")
    ax.grid(alpha=0.2, axis="y")
    ax.legend(handles=[
        Patch(facecolor=COLOR_ABOVE, alpha=0.5, label=REGIME_LABEL["ge60"]),
        Patch(facecolor=COLOR_BELOW, alpha=0.5, label=REGIME_LABEL["lt60"]),
        Line2D([0], [0], color="black", linestyle="--", marker="o", label="lift (Δ means)"),
    ], fontsize=8)

fig.suptitle("View 4 — total_points by GW block & regime, with lift overlay  (minutes > 0; outliers hidden)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### View 5 — floor / median / ceiling (quantiles)

**FPL question:** What are the **floor (P10)**, robust centre (median), and **ceiling (P90)** of each regime — and are they steady?

**Why it matters / when relevant:** The haul-proof alternative to the mean. The median is the stable centre (dead flat here = the cleanest stability evidence in the notebook); **P10 is your set-and-forget floor**, **P90 is your captaincy/differential ceiling**. Reach for this — not the mean — when choosing between a steady player and a volatile one.

In [ ]:
# View 5 — Floor (P10) / median / ceiling (P90) per regime over GW blocks, faceted by position.
# Robust to hauls: the median line is the stable centre; the band spans floor↔ceiling.
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
for i, pos in enumerate(POSITIONS):
    ax = axes.flatten()[i]
    pos_df = played[played["position"] == pos]
    for regime in ("ge60", "lt60"):
        sub = pos_df[pos_df["regime"] == regime].copy()
        sub["tp"] = sub["total_points"].astype(float)
        grp = sub.groupby("block")["tp"]
        med = grp.median().reindex(BLOCKS)
        p10 = grp.quantile(0.10).reindex(BLOCKS)
        p90 = grp.quantile(0.90).reindex(BLOCKS)
        ax.plot(BLOCKS, med, marker="o", linewidth=1.8, color=REGIME_COLOR[regime],
                label=f"{REGIME_LABEL[regime]} median", zorder=3)
        ax.fill_between(BLOCKS, p10, p90, color=REGIME_COLOR[regime], alpha=0.15,
                        zorder=1, label=f"{REGIME_LABEL[regime]} P10–P90")
    ax.set_title(pos, fontsize=13, fontweight="bold")
    ax.set_ylabel("total_points")
    ax.set_xticks(BLOCKS)
    ax.set_xticklabels([BLOCK_LABELS[b] for b in BLOCKS], rotation=45, fontsize=8)
    ax.set_xlabel("gameweek block")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
fig.suptitle("View 5 — floor (P10) · median · ceiling (P90) per regime over GW blocks  (minutes > 0)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### View 6 — hit rates (haul & blank frequency)

**FPL question:** How often does each regime **haul (≥10 pts)** versus **blank (≤2 pts)**?

**Why it matters / when relevant:** The frequency framing that zero-inflated data demands. **Haul-rate is the captaincy statistic** (you captain for ceiling *frequency*); **blank-rate is the set-and-forget statistic** (you avoid floor risk). The takeaway: a sub-60 cameo blanks ~95% of the time and essentially never hauls — useless for either role.

In [ ]:
# View 6 — Hit rates over GW blocks: haul = P(total_points ≥ 10), blank = P(≤ 2). Faceted by position.
# Solid = haul rate (ceiling frequency), dashed = blank rate (floor risk); colour = regime.
HAUL, BLANK = 10, 2
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
for i, pos in enumerate(POSITIONS):
    ax = axes.flatten()[i]
    pos_df = played[played["position"] == pos]
    for regime in ("ge60", "lt60"):
        grp = pos_df[pos_df["regime"] == regime].groupby("block")["total_points"]
        haul = grp.apply(lambda s: (s.astype(float) >= HAUL).mean() * 100).reindex(BLOCKS)
        blank = grp.apply(lambda s: (s.astype(float) <= BLANK).mean() * 100).reindex(BLOCKS)
        ax.plot(BLOCKS, haul, "-o", color=REGIME_COLOR[regime],
                label=f"{REGIME_LABEL[regime]} haul% (≥{HAUL})")
        ax.plot(BLOCKS, blank, "--x", color=REGIME_COLOR[regime], alpha=0.7,
                label=f"{REGIME_LABEL[regime]} blank% (≤{BLANK})")
    ax.set_title(pos, fontsize=13, fontweight="bold")
    ax.set_ylabel("% of appearances")
    ax.set_ylim(0, 100)
    ax.set_xticks(BLOCKS)
    ax.set_xticklabels([BLOCK_LABELS[b] for b in BLOCKS], rotation=45, fontsize=8)
    ax.set_xlabel("gameweek block")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=7, ncol=2)
fig.suptitle("View 6 — haul rate (≥10) and blank rate (≤2) per regime over GW blocks  (minutes > 0)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### View 7 — ECDF + stochastic dominance

**FPL question:** Does ≥60 beat `<60` at *every* points level (stochastic dominance), not just on average?

**Why it matters / when relevant:** The distribution-free, outlier-proof verdict — no symmetry or equal-variance assumed. The annotated `P(≥60 ≥ <60) ≈ 0.82–0.91` reads as "a random full-game return beats a random cameo ~85% of the time." Use this when you need a **robust yes/no that the populations genuinely differ**, independent of any single summary statistic.

In [ ]:
# View 7 — ECDF of total_points per regime, faceted by position, with the common-language
# effect size P(≥60 ≥ <60) annotated. Distribution-free; the lower/right curve is stochastically larger.
def _cles(a, b):
    """P(A > B) + 0.5·P(A = B) via the rank identity (matches Mann–Whitney; no scipy needed)."""
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    ranks = pd.Series(np.concatenate([a, b])).rank().to_numpy()
    u = ranks[:len(a)].sum() - len(a) * (len(a) + 1) / 2
    return u / (len(a) * len(b))

def _ecdf(s):
    x = np.sort(np.asarray(s, dtype=float))
    return x, np.arange(1, len(x) + 1) / len(x)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for i, pos in enumerate(POSITIONS):
    ax = axes.flatten()[i]
    pos_df = played[played["position"] == pos]
    a = pos_df[pos_df["regime"] == "ge60"]["total_points"]
    b = pos_df[pos_df["regime"] == "lt60"]["total_points"]
    for regime, vals in (("ge60", a), ("lt60", b)):
        x, y = _ecdf(vals)
        ax.step(x, y, where="post", color=REGIME_COLOR[regime], linewidth=1.8,
                label=f"{REGIME_LABEL[regime]} (n={len(vals):,})")
    P = _cles(a, b)
    ax.set_title(f"{pos}   P(≥60 ≥ <60) = {P:.2f}", fontsize=12, fontweight="bold")
    ax.set_xlabel("total_points")
    ax.set_ylabel("cumulative share ≤ x")
    ax.set_xlim(-3, 20)
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8, loc="lower right")
fig.suptitle("View 7 — ECDF by regime + stochastic dominance  (minutes > 0; P = 0.5 means no difference)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()